# BRGC Revision — Notebook 03: Embedding baselines (GPU)

**Purpose (Reviewer 1, comment 6).** Are classic embedding/GNN models — **TransE, RotatE, CompGCN** — also dominated by endpoint statistics on these benchmarks? We train each with PyKEEN and evaluate them on the **relation-prediction** task (h,?,t) under the same filtered ranking, so they sit on one axis with the paper's training-free baselines and the DistilBERT models.

**Attach:** `brgc-kg-splits`. **GPU on.** Run **once per dataset**. Drop CompGCN if a session is tight on time.

Send me `kge_relation_prediction_<DATASET>.csv`; I add an embedding-baseline column/row and the comparison text.

In [ ]:
# === Setup: PyKEEN + GPU ===
import subprocess, sys, torch
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError("Enable a Kaggle GPU.")

# PyKEEN 1.10.2 was built against an OLDER class_resolver (the typing signature of its
# FunctionResolver generic changed in a later class_resolver release). An unpinned
# `pip install pykeen` on Kaggle's current base image pulls the LATEST class_resolver and
# pykeen then crashes at import time with:
#   TypeError: Too few arguments for <class 'class_resolver.func.FunctionResolver'>...
# Fix: pin class_resolver to 0.4.3, the version current on the day pykeen 1.10.2 was released,
# and install both together so pip resolves them consistently regardless of what Kaggle
# preinstalled.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "pykeen==1.10.2", "class-resolver==0.4.3"])

import importlib.metadata as _im
def _ver(pkg):
    try:
        return _im.version(pkg)
    except Exception:
        return "unknown"

import pykeen
from pykeen.pipeline import pipeline  # import here (not just in the training cell) so a version
                                       # problem fails loudly in THIS cell, before any GPU time is spent
print(f"pykeen {_ver('pykeen')} + class_resolver {_ver('class_resolver')} imported OK")

from pathlib import Path
INPUT_ROOT=Path("/kaggle/input"); OUT=Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
def find_splits(name):
    for c in INPUT_ROOT.rglob(name):
        if c.is_dir() and all((c/s).exists() for s in ["train.txt","valid.txt","test.txt"]): return c
    return None

In [ ]:
# === CHOOSE dataset + models for THIS session ===
# Attach: brgc-kg-splits. GPU on. Run ONCE PER DATASET.
# Approx T4 cost (100 epochs): TransE/RotatE ~15-40 min (CoDEx) to ~1-2h (FB15k-237); CompGCN ~2-4x that.
DATASET_NAME = "CoDEx-S"                 # "CoDEx-M" / "FB15k-237" in later sessions
MODELS       = ["TransE","RotatE","CompGCN"]   # drop "CompGCN" if time is tight
NUM_EPOCHS   = 100
EMB_DIM      = 256
ds_dir = find_splits(DATASET_NAME); assert ds_dir, f"{DATASET_NAME} not attached"
print("dataset:", ds_dir, "| models:", MODELS)

In [ ]:
# === Train each KGE model and evaluate RELATION prediction (h,?,t) with filtering ===
import numpy as np, pandas as pd, json, time, collections, torch
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline

def load(p):
    import numpy as np
    return np.array([l.strip().split("\t") for l in open(p,encoding="utf-8") if l.strip()], dtype=str)

tr = load(ds_dir/"train.txt"); va = load(ds_dir/"valid.txt"); te = load(ds_dir/"test.txt")
tf_train = TriplesFactory.from_labeled_triples(tr)
tf_valid = TriplesFactory.from_labeled_triples(va, entity_to_id=tf_train.entity_to_id, relation_to_id=tf_train.relation_to_id)
tf_test  = TriplesFactory.from_labeled_triples(te, entity_to_id=tf_train.entity_to_id, relation_to_id=tf_train.relation_to_id)

# CompGCN's encoder does message passing over BOTH edge directions and asserts
# `triples_factory.create_inverse_triples` on whatever factory is passed as `training=`.
# Build a second training factory with inverse triples enabled, same entity/relation ids,
# used ONLY for CompGCN. Validation/testing stay on the ORIGINAL (non-inverse) factories for
# every model, so our custom relation-prediction eval below never sees synthetic inverse
# relations (which aren't in R_ids and would break the filtering/indexing logic).
tf_train_inv = TriplesFactory.from_labeled_triples(
    tr, entity_to_id=tf_train.entity_to_id, relation_to_id=tf_train.relation_to_id,
    create_inverse_triples=True,
)

R_ids = np.array(sorted(tf_train.relation_to_id.values()))
# known-true relations per (h,t) over train+valid+test, for filtered relation ranking
# (always from the ORIGINAL, non-inverse factories -- see note above)
known = collections.defaultdict(set)
for arr in [tf_train.mapped_triples, tf_valid.mapped_triples, tf_test.mapped_triples]:
    for h,r,t in arr.tolist(): known[(h,t)].add(r)

def relation_filtered_mrr(model, mapped_test, device):
    model.eval()
    rr=[]; h1=[]
    allR = torch.as_tensor(R_ids, device=device)
    with torch.no_grad():
        for h,r,t in mapped_test.tolist():
            hh = torch.full((len(R_ids),), h, device=device, dtype=torch.long)
            tt = torch.full((len(R_ids),), t, device=device, dtype=torch.long)
            batch = torch.stack([hh, allR, tt], dim=1)
            scores = model.score_hrt(batch).squeeze(-1)      # higher = better
            # filter other known-true relations for this (h,t)
            kt = known[(h,t)] - {r}
            if kt:
                mask = torch.as_tensor([rid in kt for rid in R_ids], device=device)
                scores = scores.masked_fill(mask, float("-inf"))
            gold_score = scores[list(R_ids).index(r)]
            greater = (scores > gold_score).sum().item()
            equal   = (scores == gold_score).sum().item() - 1
            rank = greater + equal/2.0 + 1.0
            rr.append(1.0/rank); h1.append(1.0 if rank<=1 else 0.0)
    return float(np.mean(rr)), float(np.mean(h1))

rows=[]
device = "cuda"
for model_name in MODELS:
    t0=time.time()
    print("="*70); print("TRAIN", model_name, DATASET_NAME); print("="*70)
    training_tf = tf_train_inv if model_name == "CompGCN" else tf_train
    # use_tqdm=False: Kaggle's batch/commit execution has no live front-end to drive tqdm's
    # ipywidgets progress bars, so each bar update fails with "Error displaying widget: model
    # not found" (harmless, but it spams the output once per epoch). Progress is still printed
    # via our own per-model timing line below.
    res = pipeline(training=training_tf, validation=tf_valid, testing=tf_test, model=model_name,
                   model_kwargs=dict(embedding_dim=EMB_DIM) if model_name!="CompGCN" else None,
                   training_kwargs=dict(num_epochs=NUM_EPOCHS), random_seed=42, device=device,
                   use_tqdm=False)
    model = res.model
    mrr, h1 = relation_filtered_mrr(model, tf_test.mapped_triples, device)
    dt=(time.time()-t0)/60
    print(f"{model_name}: relation-prediction filtered MRR={mrr:.4f} Hits@1={h1:.4f} ({dt:.1f} min)")
    rows.append([DATASET_NAME, model_name, EMB_DIM, NUM_EPOCHS, round(mrr,4), round(h1,4), round(dt,1)])
    pd.DataFrame(rows, columns=["Dataset","Model","EmbDim","Epochs","RelPred_FilteredMRR","RelPred_Hits@1","Minutes"]).to_csv(OUT/f"kge_relation_prediction_{DATASET_NAME}.csv", index=False)

df=pd.DataFrame(rows, columns=["Dataset","Model","EmbDim","Epochs","RelPred_FilteredMRR","RelPred_Hits@1","Minutes"])
print(df.to_string(index=False))
print("\nCompare these to the paper's p(r|t) / naive-Bayes baselines and to B0/B2 for this dataset.")
print("If TransE/RotatE/CompGCN also sit near the endpoint-statistics level, that supports the paper's claim")
print("that the benchmark is endpoint-dominated for embedding models too (Reviewer 1, comment 6).")
